In [ ]:
import os

# Set KaggleHub cache to a directory inside /content/
os.environ["KAGGLEHUB_CACHE"] = "./content/data"

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import glob
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

#here i just will create a dict to have the classes names and the reversed version to be used later
#set to get rid of dupl
#map to get names from paths apply a fun on all paths
#index to get the num of class
classes = set(map(lambda x: x.split("\\")[-2], glob.glob(path+"/PlantVillage/train/*/*")))
class_to_idx = {class_name: list(classes).index(class_name) for class_name in list(classes)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

print(class_to_idx)
print(idx_to_class)


class adataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.images_paths= glob.glob(root_dir+"/*/*")
        self.labels= list(map(lambda x: x.split("\\")[-2], self.images_paths)) # as the labels is the folders names
        self.transform = transform

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        image_path = self.images_paths[idx]
        label = class_to_idx[self.labels[idx]] #to get it as a number not a text

        image = Image.open(image_path)

        if self.transform:
            image = self.transform(image)

        return image, label


#the dataloaders here
transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])

train_dataset = adataset(path+"/PlantVillage/train", transform= transform)
test_dataset = adataset(path+"/PlantVillage/test", transform= transform)
train_loader = DataLoader(train_dataset, batch_size= 16, shuffle= True)
test_loader = DataLoader(test_dataset, batch_size= 16, shuffle= True)

#make sure it works
print(len(train_dataset))
print(len(test_dataset))

In [ ]:
# Write your code here
from torch import nn

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        #here initilize the layers to be used in forward

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # make sure of number of input channels
        self.batch1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.batch5 = nn.BatchNorm2d(256)

        # activation function -> used after each conv layer to add non linearity
        self.relu = nn.ReLU()

        # pooling layer that is also used but after relu
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # fully connected layers which is used ether for classification or regression task that include images
        # here is the first layer which in first use the flatten features (output shape of the last conv without batch size) we could know it using print(x.shape) after last conv2d layer
        self.fc1 = nn.Linear(1024, 128) # 64 * 4 * 4 is the number of features after flatten
        self.fc2 = nn.Linear(128, 3)  # number of outputs is the number of classes

    def forward(self, x):
        x = self.pool(self.relu(self.batch1(self.conv1(x))))
        x = self.pool(self.relu(self.batch2(self.conv2(x))))
        x = self.pool(self.relu(self.batch3(self.conv3(x))))
        x = self.pool(self.relu(self.batch4(self.conv4(x))))
        x = self.pool(self.relu(self.batch5(self.conv5(x))))

        # print(x.shape) #to print the shape of the last layer conv

        x = x.flatten(start_dim = 1)
        # print("Sd")
        # print("Sd")
        # print("Sd")
        # print(x.shape)
        # print("Sd")

        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # without softmax as it isn't needed as we have nn.crossentropyloss

        return x

In [ ]:
# Write your code here
from tqdm import tqdm

def training_loop(model, dataloader, criterion, optimizer, device):
    model.train()  # training mode -> handle dropout and batchnorm
    total_loss = 0

    correct = 0
    total = 0

    for images, labels in tqdm(dataloader): # tqdm to show it as in a loading bar
        images = images.to(device)
        labels = labels.to(device)
        #labels = labels.view(-1, 1).to(device) # ->  if we had shape problems in labels we make it in (batch_size, 1)

        outputs = model(images)
        # print(outputs.shape)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # accuracy -> in classification
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validation_loop(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            #labels = labels.view(-1, 1).to(device) # ->  if we had shape problems in labels we make it in (batch_size, 1)


            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()


            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim
import torch
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNNModel().to(device)
criterion = nn.CrossEntropyLoss() # as this is multiclass classification also without sigmoid (otherwise change the criterion)
optimizer = optim.AdamW(model.parameters(), lr=0.00001)  #fine tue lr to get better results (not always)
num_epochs = 4

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = training_loop(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validation_loop(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()



In [ ]:
nn.ConvTranspose2d?

In [ ]:
# Write your code here
# Write your code here
from torch import nn

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        #here initilize the layers to be used in forward

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # make sure of number of input channels
        self.batch1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.batch2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch3 = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch4 = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding=1)
        self.batch5 = nn.BatchNorm2d(256)

        # activation function -> used after each conv layer to add non linearity
        self.relu = nn.ReLU()

        # pooling layer that is also used but after relu
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # fully connected layers which is used ether for classification or regression task that include images
        # here is the first layer which in first use the flatten features (output shape of the last conv without batch size) we could know it using print(x.shape) after last conv2d layer
        self.fc1 = nn.Linear(256 *4 *4, 128) # 64 * 4 * 4 is the number of features after flatten
        self.fc2 = nn.Linear(128, 3)  # number of outputs is the number of classes

        self.up = nn.ConvTranspose2d(32, 128, kernel_size=2, stride=2)

    def forward(self, x):
        con1 = self.pool(self.relu(self.batch1(self.conv1(x))))
        con2 = self.pool(self.relu(self.batch2(self.conv2(con1))))
        con3 = self.pool(self.relu(self.batch3(self.conv3(con2))))

        con4 = self.pool(self.relu(self.batch4(self.conv4(con3))))
        # print(con2.shape)
        up_con2 = self.up(con2)
        up_con2 = self.pool(self.pool(self.pool(up_con2)))

        x = torch.cat([up_con2, con4], dim=1)

        con5 = self.pool(self.relu(self.batch5(self.conv5(x))))

        # print(x.shape) #to print the shape of the last layer conv

        x = x.flatten(start_dim = 1)
        # print("Sd")
        # print("Sd")
        # print("Sd")
        # print(x.shape)
        # print("Sd")

        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # without softmax as it isn't needed as we have nn.crossentropyloss

        return x

In [ ]:
# Write your code here
import torch.optim as optim
import torch
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CNNModel().to(device)
criterion = nn.CrossEntropyLoss() # as this is multiclass classification also without sigmoid (otherwise change the criterion)
optimizer = optim.AdamW(model.parameters(), lr=0.00001)  #fine tue lr to get better results (not always)
num_epochs = 4

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = training_loop(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validation_loop(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o') # instead of 16 use num of epochs +1
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()



In [ ]:
# there is a somehow little improvement i think, so good :)